# Mental Rotation — Debiasing Strategies

Position bias is the dominant failure mode for 3B-scale VLMs on mental rotation.
Prior experiments show models picking one option 80-100% of the time regardless of content.

This notebook tests four strategies that attack position bias at its root:

| # | Strategy | Mechanism |
|---|---|---|
| D1 | Logit extraction + permutation avg | Extract P(A),P(B) logits; run both orderings; average |
| D2 | Binary decomposition | Ask "is this a mirror? YES/NO" per option — no A/B position |
| D3 | Feature-grounded prompt | Anchor on concrete asymmetric feature + logit extraction |
| D4 | Few-shot examples | 2 demonstration trials + elimination prompt + logit extraction |

### Prior best result

| Model | Strategy | Accuracy | p-value |
|---|---|---|---|
| Qwen2.5-VL-3B | Self-consistency k=5 (elimination) | 62.7% | 0.014 |

In [ ]:
from __future__ import annotations

import csv, json, os, random, re, time
from collections import Counter
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
from scipy import stats
from PIL import Image

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "data").is_dir() and (p / "src").is_dir()),
    Path.cwd().parent.parent,
)

LABELS = ["A", "B"]
CHANCE = 0.5
DATA_ROOT = ROOT / "data"
MANIFEST_PATH = DATA_ROOT / "assets" / "manifest.csv"
RESULTS_DIR = ROOT / "results" / "prompt_optimization" / "mental-rotation" / "debiasing"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")
print(f"Results dir:  {RESULTS_DIR}")

In [ ]:
def _detect_image_dir() -> Path:
    assets = DATA_ROOT / "assets"
    for child in sorted(assets.iterdir(), reverse=True):
        candidate = child / "visual" / "mental-rotation"
        if candidate.is_dir() and any(candidate.iterdir()):
            return candidate
    raise FileNotFoundError("No mental-rotation images found.")


def _build_image_index(directory: Path) -> dict[str, Path]:
    index: dict[str, Path] = {}
    if not directory.is_dir():
        return index
    for path in directory.iterdir():
        if path.is_file():
            index[path.stem] = path
            index[path.name] = path
    return index


def _extract_angle(item_uid: str) -> Optional[int]:
    m = re.search(r"_(\d{3})_", item_uid)
    if m:
        return int(m.group(1))
    if item_uid.endswith("-000") or "_000" in item_uid or item_uid.startswith(("Rn-000", "Rp-000")):
        return 0
    return None


IMAGE_DIR = _detect_image_dir()
IMAGE_INDEX = _build_image_index(IMAGE_DIR)
print(f"Image dir: {IMAGE_DIR} ({len(IMAGE_INDEX) // 2} images)")

In [ ]:
def load_trials() -> list[dict]:
    rows = []
    with open(MANIFEST_PATH, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            if row["task"] == "mental-rotation":
                rows.append(row)

    trials = []
    for row in rows:
        answer = str(row["answer"]).strip()
        alternatives = [a.strip() for a in row["response_alternatives"].split(",") if a.strip()]
        all_options = [answer] + alternatives
        rng = random.Random(row["item_uid"])
        rng.shuffle(all_options)
        correct_idx = all_options.index(answer)
        correct_label = LABELS[correct_idx] if correct_idx < len(LABELS) else "?"

        option_image_paths = []
        missing = False
        for option in all_options:
            path = IMAGE_INDEX.get(option.strip())
            if path is None:
                missing = True
                break
            option_image_paths.append(str(path))

        prompt_image_stem = str(row.get("prompt_image", "")).strip()
        context_image_paths = []
        if prompt_image_stem and prompt_image_stem not in {"NA", "nan", "TODO", ""}:
            path = IMAGE_INDEX.get(prompt_image_stem)
            if path is None:
                missing = True
            else:
                context_image_paths.append(str(path))

        if missing:
            continue

        trials.append({
            "item_uid": row["item_uid"],
            "trial_type": str(row.get("trial_type", "")).strip(),
            "angle": _extract_angle(row["item_uid"]),
            "full_prompt": row.get("full_prompt", ""),
            "options": all_options,
            "option_image_paths": option_image_paths,
            "context_image_paths": context_image_paths,
            "correct_label": correct_label,
        })
    return trials


TRIALS = load_trials()
print(f"Loaded {len(TRIALS)} trials")
print(f"Correct-label distribution: {Counter(t['correct_label'] for t in TRIALS)}")

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

HF_MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

SYSTEM_PROMPT = (
    "You are a visual reasoning assistant. "
    "Answer with only a single letter: A or B. Do not explain."
)

t0 = time.time()
device = "cuda" if torch.cuda.is_available() else (
    "mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu"
)
dtype = torch.bfloat16
processor = AutoProcessor.from_pretrained(HF_MODEL_ID, padding_side="left")
model = AutoModelForImageTextToText.from_pretrained(
    HF_MODEL_ID, torch_dtype=dtype, attn_implementation="sdpa",
).to(device)
model.eval()
print(f"Loaded {HF_MODEL_ID} on {device} in {time.time()-t0:.1f}s")

## Logit extraction and permutation averaging

In [ ]:
def _build_inputs(prompt_text: str, image_paths: list[str],
                   system_prompt: str | None = None):
    """Build processor inputs from prompt text and image paths."""
    pil_images = [Image.open(p).convert("RGB") for p in image_paths]

    content: list[dict] = []
    if pil_images and re.search(r"<image\d+>", prompt_text):
        parts = re.split(r"(<image\d+>)", prompt_text)
        for part in parts:
            m = re.match(r"<image(\d+)>", part)
            if m:
                idx = int(m.group(1))
                if idx < len(pil_images):
                    content.append({"type": "image", "image": pil_images[idx]})
            elif part.strip():
                content.append({"type": "text", "text": part.strip()})
    else:
        for img in pil_images:
            content.append({"type": "image", "image": img})
        content.append({"type": "text", "text": prompt_text})

    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": content})

    try:
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )

    inputs = processor(
        text=[text],
        images=pil_images if pil_images else None,
        return_tensors="pt",
        padding=True,
    ).to(device)
    return inputs


# Resolve token IDs for answer labels
TOKEN_A = processor.tokenizer.encode("A", add_special_tokens=False)[-1]
TOKEN_B = processor.tokenizer.encode("B", add_special_tokens=False)[-1]
TOKEN_YES = processor.tokenizer.encode("Yes", add_special_tokens=False)[-1]
TOKEN_NO = processor.tokenizer.encode("No", add_special_tokens=False)[-1]
TOKEN_YES_LC = processor.tokenizer.encode("yes", add_special_tokens=False)[-1]
TOKEN_NO_LC = processor.tokenizer.encode("no", add_special_tokens=False)[-1]

print(f"Token IDs — A: {TOKEN_A}, B: {TOKEN_B}, Yes: {TOKEN_YES}, No: {TOKEN_NO}")
print(f"Token IDs — yes: {TOKEN_YES_LC}, no: {TOKEN_NO_LC}")

In [ ]:
def get_logits_ab(prompt_text: str, image_paths: list[str],
                  system_prompt: str | None = None) -> tuple[float, float]:
    """Return (logit_A, logit_B) for the next token given the prompt."""
    inputs = _build_inputs(prompt_text, image_paths, system_prompt)
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits[0, -1]  # last token position
    return logits[TOKEN_A].item(), logits[TOKEN_B].item()


def get_logits_yesno(prompt_text: str, image_paths: list[str],
                     system_prompt: str | None = None) -> tuple[float, float]:
    """Return (logit_yes, logit_no) for the next token."""
    inputs = _build_inputs(prompt_text, image_paths, system_prompt)
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits[0, -1]
    yes_logit = max(logits[TOKEN_YES].item(), logits[TOKEN_YES_LC].item())
    no_logit = max(logits[TOKEN_NO].item(), logits[TOKEN_NO_LC].item())
    return yes_logit, no_logit


def generate_text(prompt_text: str, image_paths: list[str],
                  system_prompt: str | None = None,
                  max_new_tokens: int = 128,
                  do_sample: bool = False,
                  temperature: float = 1.0) -> str:
    """Standard text generation (for few-shot / comparison)."""
    inputs = _build_inputs(prompt_text, image_paths, system_prompt)
    gen_kwargs = dict(max_new_tokens=max_new_tokens)
    if do_sample:
        gen_kwargs.update(do_sample=True, temperature=temperature, top_p=0.9)
    else:
        gen_kwargs["do_sample"] = False
    input_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        output_ids = model.generate(**inputs, **gen_kwargs)
    generated_ids = output_ids[:, input_len:]
    raw = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    return raw

In [ ]:
def _binom_p(correct: int, n: int) -> float:
    if n == 0:
        return 1.0
    return stats.binomtest(correct, n, CHANCE, alternative="greater").pvalue


def parse_answer(text: str) -> Optional[str]:
    text = text.strip().upper()
    if text in ("A", "B"):
        return text
    for pat in (
        r"(?:the\s+)?(?:correct\s+)?answer\s+is\s+([A-B])\b",
        r"option\s+([A-B])\b",
        r"(?:choose|select|pick)\s+([A-B])\b",
        r"\(([A-B])\)",
    ):
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            return m.group(1).upper()
    for label in LABELS:
        if text.startswith(label):
            rest = text[len(label):]
            if not rest or rest[0] in " .),:;\n":
                return label
    for sentence in reversed(re.split(r"[.!?\n]", text)):
        for label in LABELS:
            if re.search(rf"\b{re.escape(label)}\b", sentence, re.IGNORECASE):
                return label
    return None

## Smoke test: verify logit extraction works

In [ ]:
trial = TRIALS[0]
prompt = (
    "Reference image:\n<image0>\n\n"
    "Below are two options. One is the SAME shape rotated. "
    "The other is a MIRROR IMAGE (horizontally flipped).\n\n"
    "First, identify which option looks like a mirror/flip of the reference.\n"
    "Then pick the OTHER option — the one that is simply rotated.\n\n"
    "A: <image1>\nB: <image2>\n\n"
    "Answer with one letter: A or B."
)
images = trial["context_image_paths"] + trial["option_image_paths"]

logit_a, logit_b = get_logits_ab(prompt, images, SYSTEM_PROMPT)
p_a = np.exp(logit_a) / (np.exp(logit_a) + np.exp(logit_b))
p_b = 1 - p_a

text_out = generate_text(prompt, images, SYSTEM_PROMPT, max_new_tokens=4)

print(f"Trial: {trial['item_uid']}, correct: {trial['correct_label']}")
print(f"Logit A: {logit_a:.3f}, Logit B: {logit_b:.3f}")
print(f"P(A): {p_a:.3f}, P(B): {p_b:.3f}")
print(f"Logit choice: {'A' if logit_a > logit_b else 'B'}")
print(f"Text generation: {text_out!r}")

## Strategy D1: Logit extraction + permutation averaging

For each trial, run the elimination prompt twice:
1. Original order (A=option0, B=option1)
2. Swapped order (A=option1, B=option0)

Average the logit-based probabilities across both orderings. If the model has real
spatial understanding, the correct answer's probability should survive averaging;
if it's pure position bias, the probabilities cancel to ~50%.

In [ ]:
from tqdm.notebook import tqdm
import psutil

ELIM_PROMPT_TEMPLATE = (
    "Reference image:\n<image0>\n\n"
    "Below are two options. One is the SAME shape rotated to a different angle. "
    "The other is a MIRROR IMAGE (horizontally flipped).\n\n"
    "First, identify which option looks like a mirror/flip of the reference.\n"
    "Then pick the OTHER option — the one that is simply rotated.\n\n"
    "A: <image1>\nB: <image2>\n\n"
    "Answer with one letter: A or B."
)

proc = psutil.Process(os.getpid())


def run_d1_logit_perm(trials: list[dict]) -> list[dict]:
    results = []
    for trial in tqdm(trials, desc="D1_logit_perm"):
        ctx = trial["context_image_paths"]
        opt = trial["option_image_paths"]

        # Pass 1: original order
        imgs_orig = ctx + opt
        la1, lb1 = get_logits_ab(ELIM_PROMPT_TEMPLATE, imgs_orig, SYSTEM_PROMPT)

        # Pass 2: swapped order (option images reversed)
        imgs_swap = ctx + [opt[1], opt[0]]
        la2, lb2 = get_logits_ab(ELIM_PROMPT_TEMPLATE, imgs_swap, SYSTEM_PROMPT)

        # In swapped pass, "A" refers to what was originally B and vice versa.
        # So the logit for original-A from pass 2 is lb2, and original-B is la2.
        avg_logit_origA = (la1 + lb2) / 2
        avg_logit_origB = (lb1 + la2) / 2

        p_origA = np.exp(avg_logit_origA) / (np.exp(avg_logit_origA) + np.exp(avg_logit_origB))

        predicted = "A" if p_origA > 0.5 else "B"

        results.append({
            "item_uid": trial["item_uid"],
            "trial_type": trial["trial_type"],
            "angle": trial["angle"],
            "correct_label": trial["correct_label"],
            "predicted_label": predicted,
            "is_correct": predicted == trial["correct_label"],
            "p_origA": float(p_origA),
            "logit_A_pass1": la1, "logit_B_pass1": lb1,
            "logit_A_pass2": la2, "logit_B_pass2": lb2,
            "strategy": "D1_logit_perm",
        })
    return results


t0 = time.time()
d1_results = run_d1_logit_perm(TRIALS)
d1_time = time.time() - t0
d1_correct = sum(r["is_correct"] for r in d1_results)
rss = proc.memory_info().rss / 1024**3
print(f"D1_logit_perm: {d1_correct}/{len(TRIALS)} correct "
      f"({d1_correct/len(TRIALS):.1%}), {d1_time:.0f}s, RSS={rss:.1f}GB")

In [ ]:
# Also run D1 without permutation (single pass) for comparison
def run_d1_logit_single(trials: list[dict]) -> list[dict]:
    results = []
    for trial in tqdm(trials, desc="D1_logit_single"):
        ctx = trial["context_image_paths"]
        opt = trial["option_image_paths"]
        imgs = ctx + opt
        la, lb = get_logits_ab(ELIM_PROMPT_TEMPLATE, imgs, SYSTEM_PROMPT)
        p_a = np.exp(la) / (np.exp(la) + np.exp(lb))
        predicted = "A" if la > lb else "B"
        results.append({
            "item_uid": trial["item_uid"],
            "trial_type": trial["trial_type"],
            "angle": trial["angle"],
            "correct_label": trial["correct_label"],
            "predicted_label": predicted,
            "is_correct": predicted == trial["correct_label"],
            "p_A": float(p_a),
            "logit_A": la, "logit_B": lb,
            "strategy": "D1_logit_single",
        })
    return results


t0 = time.time()
d1s_results = run_d1_logit_single(TRIALS)
d1s_time = time.time() - t0
d1s_correct = sum(r["is_correct"] for r in d1s_results)
print(f"D1_logit_single: {d1s_correct}/{len(TRIALS)} correct "
      f"({d1s_correct/len(TRIALS):.1%}), {d1s_time:.0f}s")

## Strategy D2: Binary decomposition

Eliminate A/B framing entirely. For each option, ask independently:
"Is this image a mirror (horizontally flipped) version of the reference? YES or NO."

The option with the lower mirror probability is the rotation.

In [ ]:
BINARY_SYSTEM = (
    "You are a visual reasoning assistant. "
    "Answer with only YES or NO. Do not explain."
)

BINARY_PROMPT = (
    "Reference shape:\n<image0>\n\n"
    "Test shape:\n<image1>\n\n"
    "Is the test shape a MIRROR IMAGE (horizontally flipped) of the reference?\n"
    "Answer YES or NO."
)


def run_d2_binary(trials: list[dict]) -> list[dict]:
    results = []
    for trial in tqdm(trials, desc="D2_binary"):
        ctx = trial["context_image_paths"]
        opt = trial["option_image_paths"]

        # Ask about option A (index 0)
        yes_a, no_a = get_logits_yesno(BINARY_PROMPT, ctx + [opt[0]], BINARY_SYSTEM)
        # Ask about option B (index 1)
        yes_b, no_b = get_logits_yesno(BINARY_PROMPT, ctx + [opt[1]], BINARY_SYSTEM)

        # "Mirror score" = how much the model thinks this is a mirror
        mirror_score_a = yes_a - no_a
        mirror_score_b = yes_b - no_b

        # The rotation is the one LESS likely to be a mirror
        predicted = "A" if mirror_score_a < mirror_score_b else "B"

        results.append({
            "item_uid": trial["item_uid"],
            "trial_type": trial["trial_type"],
            "angle": trial["angle"],
            "correct_label": trial["correct_label"],
            "predicted_label": predicted,
            "is_correct": predicted == trial["correct_label"],
            "mirror_score_A": mirror_score_a,
            "mirror_score_B": mirror_score_b,
            "yes_A": yes_a, "no_A": no_a,
            "yes_B": yes_b, "no_B": no_b,
            "strategy": "D2_binary",
        })
    return results


t0 = time.time()
d2_results = run_d2_binary(TRIALS)
d2_time = time.time() - t0
d2_correct = sum(r["is_correct"] for r in d2_results)
print(f"D2_binary: {d2_correct}/{len(TRIALS)} correct "
      f"({d2_correct/len(TRIALS):.1%}), {d2_time:.0f}s, RSS={proc.memory_info().rss/1024**3:.1f}GB")

In [ ]:
# D2 variant: ask "Is this the SAME shape rotated?" (positive framing)
BINARY_SAME_PROMPT = (
    "Reference shape:\n<image0>\n\n"
    "Test shape:\n<image1>\n\n"
    "Is the test shape the SAME shape as the reference, just rotated to a different angle?\n"
    "Answer YES or NO."
)


def run_d2_binary_same(trials: list[dict]) -> list[dict]:
    results = []
    for trial in tqdm(trials, desc="D2_binary_same"):
        ctx = trial["context_image_paths"]
        opt = trial["option_image_paths"]

        yes_a, no_a = get_logits_yesno(BINARY_SAME_PROMPT, ctx + [opt[0]], BINARY_SYSTEM)
        yes_b, no_b = get_logits_yesno(BINARY_SAME_PROMPT, ctx + [opt[1]], BINARY_SYSTEM)

        # "Same score" = how much the model thinks this IS the rotation
        same_score_a = yes_a - no_a
        same_score_b = yes_b - no_b

        # Pick the one MORE likely to be the same/rotated
        predicted = "A" if same_score_a > same_score_b else "B"

        results.append({
            "item_uid": trial["item_uid"],
            "trial_type": trial["trial_type"],
            "angle": trial["angle"],
            "correct_label": trial["correct_label"],
            "predicted_label": predicted,
            "is_correct": predicted == trial["correct_label"],
            "same_score_A": same_score_a,
            "same_score_B": same_score_b,
            "strategy": "D2_binary_same",
        })
    return results


t0 = time.time()
d2s_results = run_d2_binary_same(TRIALS)
d2s_time = time.time() - t0
d2s_correct = sum(r["is_correct"] for r in d2s_results)
print(f"D2_binary_same: {d2s_correct}/{len(TRIALS)} correct "
      f"({d2s_correct/len(TRIALS):.1%}), {d2s_time:.0f}s")

## Strategy D3: Feature-grounded prompt

Reduce the task from holistic shape comparison to a single local feature check.
Uses logit extraction + permutation averaging.

In [ ]:
FEATURE_PROMPT = (
    "Reference shape:\n<image0>\n\n"
    "Option A:\n<image1>\nOption B:\n<image2>\n\n"
    "Focus on the most prominent protruding part of the reference shape.\n"
    "Note which SIDE it is on (left or right).\n\n"
    "One option is the same shape rotated — its protrusion is on the SAME relative side.\n"
    "The other is a mirror image — its protrusion is on the OPPOSITE side.\n\n"
    "Which option has the protrusion on the SAME side as the reference?\n"
    "Answer with one letter: A or B."
)

FEATURE_SYSTEM = (
    "You are a visual reasoning assistant. "
    "Answer with only a single letter: A or B. Do not explain."
)


def run_d3_feature(trials: list[dict]) -> list[dict]:
    results = []
    for trial in tqdm(trials, desc="D3_feature"):
        ctx = trial["context_image_paths"]
        opt = trial["option_image_paths"]

        # Pass 1: original order
        la1, lb1 = get_logits_ab(FEATURE_PROMPT, ctx + opt, FEATURE_SYSTEM)

        # Pass 2: swapped
        la2, lb2 = get_logits_ab(FEATURE_PROMPT, ctx + [opt[1], opt[0]], FEATURE_SYSTEM)

        avg_logit_A = (la1 + lb2) / 2
        avg_logit_B = (lb1 + la2) / 2
        p_A = np.exp(avg_logit_A) / (np.exp(avg_logit_A) + np.exp(avg_logit_B))
        predicted = "A" if p_A > 0.5 else "B"

        results.append({
            "item_uid": trial["item_uid"],
            "trial_type": trial["trial_type"],
            "angle": trial["angle"],
            "correct_label": trial["correct_label"],
            "predicted_label": predicted,
            "is_correct": predicted == trial["correct_label"],
            "p_A": float(p_A),
            "strategy": "D3_feature",
        })
    return results


t0 = time.time()
d3_results = run_d3_feature(TRIALS)
d3_time = time.time() - t0
d3_correct = sum(r["is_correct"] for r in d3_results)
print(f"D3_feature: {d3_correct}/{len(TRIALS)} correct "
      f"({d3_correct/len(TRIALS):.1%}), {d3_time:.0f}s, RSS={proc.memory_info().rss/1024**3:.1f}GB")

## Strategy D4: Few-shot examples

Provide 2 demonstration trials (held out from evaluation) showing what rotation
vs mirror looks like, then ask the elimination question with logit extraction +
permutation averaging.

In [ ]:
# Select 2 demo trials and hold them out
# Pick one where correct=A and one where correct=B for balance
demo_a = next(t for t in TRIALS if t["correct_label"] == "A")
demo_b = next(t for t in TRIALS if t["correct_label"] == "B" and t != demo_a)
DEMO_TRIALS = [demo_a, demo_b]
EVAL_TRIALS = [t for t in TRIALS if t not in DEMO_TRIALS]
print(f"Demo trials: {[d['item_uid'] for d in DEMO_TRIALS]}")
print(f"Eval trials: {len(EVAL_TRIALS)} (held out {len(DEMO_TRIALS)})")

In [ ]:
def _build_fewshot_inputs(trial: dict, demo_trials: list[dict],
                          system_prompt: str | None = None):
    """Build a multi-turn conversation with demo examples + the real trial."""
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    all_pil_images = []
    img_counter = 0

    for demo in demo_trials:
        d_ctx = demo["context_image_paths"]
        d_opt = demo["option_image_paths"]
        d_imgs = d_ctx + d_opt

        user_content = []
        for p in d_imgs:
            pil = Image.open(p).convert("RGB")
            all_pil_images.append(pil)
            user_content.append({"type": "image", "image": pil})

        user_content.append({"type": "text", "text": (
            "Reference image (first), then Option A and Option B.\n"
            "One option is the same shape rotated. The other is a mirror image (flipped).\n"
            "Which option is the rotated copy (not flipped)? Answer A or B."
        )})
        messages.append({"role": "user", "content": user_content})
        messages.append({"role": "assistant", "content": demo["correct_label"]})

    # Now the actual trial
    t_ctx = trial["context_image_paths"]
    t_opt = trial["option_image_paths"]
    t_imgs = t_ctx + t_opt

    user_content = []
    for p in t_imgs:
        pil = Image.open(p).convert("RGB")
        all_pil_images.append(pil)
        user_content.append({"type": "image", "image": pil})

    user_content.append({"type": "text", "text": (
        "Reference image (first), then Option A and Option B.\n"
        "One option is the same shape rotated. The other is a mirror image (flipped).\n"
        "Which option is the rotated copy (not flipped)? Answer A or B."
    )})
    messages.append({"role": "user", "content": user_content})

    try:
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )

    inputs = processor(
        text=[text],
        images=all_pil_images if all_pil_images else None,
        return_tensors="pt",
        padding=True,
    ).to(device)
    return inputs


def get_logits_ab_fewshot(trial: dict, demo_trials: list[dict],
                          system_prompt: str | None = None) -> tuple[float, float]:
    inputs = _build_fewshot_inputs(trial, demo_trials, system_prompt)
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits[0, -1]
    return logits[TOKEN_A].item(), logits[TOKEN_B].item()


def run_d4_fewshot(eval_trials: list[dict], demo_trials: list[dict]) -> list[dict]:
    results = []
    for trial in tqdm(eval_trials, desc="D4_fewshot"):
        # Pass 1: original order
        la1, lb1 = get_logits_ab_fewshot(trial, demo_trials, SYSTEM_PROMPT)

        # Pass 2: swap the test trial's options
        swapped_trial = dict(trial)
        swapped_trial["option_image_paths"] = [
            trial["option_image_paths"][1],
            trial["option_image_paths"][0],
        ]
        la2, lb2 = get_logits_ab_fewshot(swapped_trial, demo_trials, SYSTEM_PROMPT)

        avg_logit_A = (la1 + lb2) / 2
        avg_logit_B = (lb1 + la2) / 2
        p_A = np.exp(avg_logit_A) / (np.exp(avg_logit_A) + np.exp(avg_logit_B))
        predicted = "A" if p_A > 0.5 else "B"

        results.append({
            "item_uid": trial["item_uid"],
            "trial_type": trial["trial_type"],
            "angle": trial["angle"],
            "correct_label": trial["correct_label"],
            "predicted_label": predicted,
            "is_correct": predicted == trial["correct_label"],
            "p_A": float(p_A),
            "strategy": "D4_fewshot",
        })
    return results


t0 = time.time()
d4_results = run_d4_fewshot(EVAL_TRIALS, DEMO_TRIALS)
d4_time = time.time() - t0
d4_correct = sum(r["is_correct"] for r in d4_results)
print(f"D4_fewshot: {d4_correct}/{len(EVAL_TRIALS)} correct "
      f"({d4_correct/len(EVAL_TRIALS):.1%}), {d4_time:.0f}s, RSS={proc.memory_info().rss/1024**3:.1f}GB")

## Summary Results

In [ ]:
all_results = {
    "D1_logit_perm": d1_results,
    "D1_logit_single": d1s_results,
    "D2_binary_mirror": d2_results,
    "D2_binary_same": d2s_results,
    "D3_feature": d3_results,
    "D4_fewshot": d4_results,
}

summary_rows = []
for name, results in all_results.items():
    n = len(results)
    correct = sum(r["is_correct"] for r in results)
    parsed = sum(1 for r in results if r["predicted_label"] is not None)
    acc = correct / n if n else 0
    p_val = _binom_p(correct, n)
    summary_rows.append({
        "Strategy": name,
        "N": n,
        "Correct": correct,
        "Accuracy": f"{acc:.1%}",
        "p-value": f"{p_val:.4f}",
        "Sig (p<.05)": "Yes" if p_val < 0.05 else "No",
    })

df_summary = pd.DataFrame(summary_rows).sort_values("Correct", ascending=False)
print(df_summary.to_string(index=False))

## Response Bias Analysis

In [ ]:
bias_rows = []
for name, results in all_results.items():
    parsed = [r for r in results if r["predicted_label"] is not None]
    if not parsed:
        continue
    counts = Counter(r["predicted_label"] for r in parsed)
    n = len(parsed)
    a_count = counts.get("A", 0)
    b_count = counts.get("B", 0)
    expected = n / 2
    chi2 = sum((v - expected) ** 2 / expected for v in [a_count, b_count])
    p_val = 1 - stats.chi2.cdf(chi2, df=1)
    bias_rows.append({
        "Strategy": name,
        "N": n,
        "A count": a_count,
        "B count": b_count,
        "A %": f"{a_count/n:.0%}",
        "chi2 p": f"{p_val:.3f}",
        "Biased": "Yes" if p_val < 0.05 else "No",
    })

print(pd.DataFrame(bias_rows).to_string(index=False))

## Logit Distribution Analysis

In [ ]:
# D1 permutation: how much do the two passes agree?
print("=== D1 Logit Permutation Analysis ===")
probs = [r["p_origA"] for r in d1_results]
print(f"Mean P(original A): {np.mean(probs):.3f} (0.5 = no bias)")
print(f"Std P(original A):  {np.std(probs):.3f}")
print(f"Trials where P(A) > 0.6: {sum(1 for p in probs if p > 0.6)}")
print(f"Trials where P(A) < 0.4: {sum(1 for p in probs if p < 0.4)}")
print(f"Trials where 0.45 < P(A) < 0.55 (uncertain): {sum(1 for p in probs if 0.45 < p < 0.55)}")

# Check if pass1 and pass2 disagree
agree = 0
for r in d1_results:
    p1_choice = "A" if r["logit_A_pass1"] > r["logit_B_pass1"] else "B"
    p2_choice = "B" if r["logit_A_pass2"] > r["logit_B_pass2"] else "A"  # swapped
    if p1_choice == p2_choice:
        agree += 1
print(f"\nPass 1 & Pass 2 agree on original label: {agree}/{len(d1_results)} ({agree/len(d1_results):.0%})")

print("\n=== D2 Binary Mirror Scores ===")
diffs = [r["mirror_score_A"] - r["mirror_score_B"] for r in d2_results]
print(f"Mean (mirror_A - mirror_B): {np.mean(diffs):.3f}")
print(f"Std: {np.std(diffs):.3f}")
print(f"Trials where diff > 1.0 (confident): {sum(1 for d in diffs if abs(d) > 1.0)}")
print(f"Trials where diff < 0.5 (uncertain): {sum(1 for d in diffs if abs(d) < 0.5)}")

## Accuracy by Trial Type

In [ ]:
type_rows = []
for name, results in all_results.items():
    for r in results:
        type_rows.append({
            "Strategy": name,
            "Type": "2D silhouettes" if r["trial_type"] == "2" else "3D blocks",
            "is_correct": r["is_correct"],
        })

df_type = pd.DataFrame(type_rows)
pivot = df_type.pivot_table(index="Strategy", columns="Type", values="is_correct", aggfunc="mean")
print(pivot.to_string())

## Comparison vs Prior Best Results

In [ ]:
prior_best = {
    "Qwen S6_selfcons_k5 (text gen)": {"acc": 0.627, "p": 0.014, "method": "Self-consistency k=5"},
    "Qwen S2_elimination (text gen)": {"acc": 0.590, "p": 0.062, "method": "Elimination greedy"},
    "SpaceThinker S2_imgfirst (text gen)": {"acc": 0.590, "p": 0.062, "method": "Image-first elimination"},
}

comparison_rows = []
for name, info in prior_best.items():
    comparison_rows.append({
        "Strategy": name,
        "Accuracy": f"{info['acc']:.1%}",
        "p-value": f"{info['p']:.4f}",
        "Sig": "Yes" if info['p'] < 0.05 else "No",
        "Method": info['method'],
    })

for name, results in all_results.items():
    n = len(results)
    correct = sum(r["is_correct"] for r in results)
    acc = correct / n
    p_val = _binom_p(correct, n)
    comparison_rows.append({
        "Strategy": f"{name} (logit)",
        "Accuracy": f"{acc:.1%}",
        "p-value": f"{p_val:.4f}",
        "Sig": "Yes" if p_val < 0.05 else "No",
        "Method": "Logit-based debiasing",
    })

df_comp = pd.DataFrame(comparison_rows).sort_values("Accuracy", ascending=False)
print(df_comp.to_string(index=False))

## Save Results

In [ ]:
for name, results in all_results.items():
    path = RESULTS_DIR / f"{name}.csv"
    df = pd.DataFrame(results)
    df.to_csv(path, index=False)
    print(f"Saved {path.name} ({len(results)} rows)")

summary_path = RESULTS_DIR / "summary.csv"
df_summary.to_csv(summary_path, index=False)
print(f"Saved {summary_path.name}")

## Conclusions

### Accuracy summary (Qwen2.5-VL-3B, 83 trials)

| Strategy | N | Correct | Accuracy | p-value | Sig (p<.05) |
|---|---|---|---|---|---|
| **D2_binary_same** | 83 | 49 | **59.0%** | 0.062 | No |
| D1_logit_single | 83 | 47 | 56.6% | 0.136 | No |
| D1_logit_perm | 83 | 43 | 51.8% | 0.413 | No |
| D2_binary_mirror | 83 | 42 | 50.6% | 0.500 | No |
| D3_feature | 83 | 42 | 50.6% | 0.500 | No |
| D4_fewshot | 81 | 40 | 49.4% | 0.588 | No |

### Response bias

| Strategy | A count | B count | A % | chi2 p | Biased? |
|---|---|---|---|---|---|
| D1_logit_single | 6 | 77 | 7% | 0.000 | Yes |
| D4_fewshot | 14 | 67 | 17% | 0.000 | Yes |
| D1_logit_perm | 20 | 63 | 24% | 0.000 | Yes |
| D2_binary_mirror | 25 | 58 | 30% | 0.000 | Yes |
| D2_binary_same | 30 | 53 | 36% | 0.012 | Yes |
| **D3_feature** | **33** | **50** | **40%** | **0.062** | **No** |

### Permutation analysis (D1_logit_perm)

| Metric | Value |
|---|---|
| Mean P(original A) | 0.498 (0.5 = no bias) |
| Std P(original A) | 0.019 |
| Trials P(A) > 0.6 | 0/83 |
| Trials P(A) < 0.4 | 0/83 |
| Uncertain (0.45-0.55) | **81/83 (98%)** |
| Pass1 and Pass2 agree | 12/83 (14%) |

### Accuracy by trial type

| Strategy | 2D silhouettes | 3D blocks |
|---|---|---|
| D1_logit_perm | 51.0% | 52.9% |
| D1_logit_single | 63.3% | 47.1% |
| D2_binary_mirror | 59.2% | 38.2% |
| **D2_binary_same** | **59.2%** | **58.8%** |
| D3_feature | 49.0% | 52.9% |
| D4_fewshot | 48.9% | 50.0% |

### Comparison vs all prior results

| Strategy | Accuracy | p-value | Sig? | Method |
|---|---|---|---|---|
| Qwen S6_selfcons_k5 | **62.7%** | **0.014** | **Yes** | Text gen, self-consistency |
| **D2_binary_same** | **59.0%** | **0.062** | No | Logit, binary decomposition |
| Qwen S2_elimination | 59.0% | 0.062 | No | Text gen, greedy |
| SpaceThinker S2_imgfirst | 59.0% | 0.062 | No | Text gen, greedy |
| D1_logit_single | 56.6% | 0.136 | No | Logit, single pass |
| D1_logit_perm | 51.8% | 0.413 | No | Logit, permutation averaged |
| D2_binary_mirror | 50.6% | 0.500 | No | Logit, binary decomposition |
| D3_feature | 50.6% | 0.500 | No | Logit, perm averaged |
| D4_fewshot | 49.4% | 0.588 | No | Logit, few-shot + perm avg |

---

## Analysis: Implications for Qwen, SpaceThinker, and VLM Benchmarking

### The key finding: permutation averaging reveals near-chance performance

The D1_logit_perm result (~52%) is the most important finding across all mental rotation
experiments. When position bias is algebraically cancelled via permutation averaging, Qwen2.5-VL-3B
falls to near chance. This means:

- The 56-63% accuracy in prior experiments comes largely from the model's B-bias accidentally
  correlating with the answer distribution (49 of 83 trials have B as correct = 59%).
- The self-consistency result (62.7%, p=0.014) was likely a statistical artifact of bias sampling
  combined with unbalanced labels, not genuine spatial reasoning.

### How to apply these findings

#### 1. For Qwen base (and similar VLMs)

**Logit-based evaluation is essential for honest benchmarking.** Text generation accuracy is inflated
by bias-label correlation. Any benchmark with unbalanced correct-answer distributions will have this
problem. Recommendation: always report both raw accuracy AND permutation-averaged logit accuracy.

**Self-consistency works not because of reasoning, but because of variance.** The k=5 sampling
introduces different biases per sample, and majority voting acts as a weak debiasing mechanism.
This suggests that **logit-based self-consistency** (averaging logit probabilities over k
temperature-sampled forward passes, then permutation-averaging) could be a stronger approach —
combining the benefits of both methods.

**Binary decomposition may be more honest.** If D2 results come back near 50%, it confirms the
model genuinely cannot distinguish rotation from mirror. If D2 is significantly above chance, it
means the A/B framing itself was hurting the model, and per-option evaluation is preferable.

#### 2. For SpaceThinker and fine-tuned VLMs

**Spatial LoRA did not help chirality.** SpaceThinker's fine-tuning on distance/position tasks
does not transfer to mirror-image discrimination. A targeted fine-tuning approach would need:
- Training data specifically about chirality (left-hand vs right-hand shapes)
- GRPO/DPO with reward signal from rotation vs mirror classification
- The reward should use **permutation-averaged logit accuracy**, not raw text output accuracy,
  to avoid training the model's bias rather than its perceptual skill

**VL-Thinking mode is counterproductive at 3B scale.** Extended reasoning traces introduce more
bias, not less. For fine-tuning: keep the model constrained to short outputs for binary
discrimination tasks. The thinking overhead (5-15x slower) yields strictly worse accuracy.

#### 3. For the benchmark pipeline generally

**Use logit extraction as the default evaluation method** for all forced-choice tasks (mental
rotation, shape bias, etc.). Benefits:
- Faster: ~3s vs ~12s per trial
- More informative: gives continuous confidence scores, not just discrete labels
- No parsing issues: avoids the failure modes of text extraction
- Enables principled debiasing via permutation averaging

**Balance the answer distribution.** The current 34A/49B split means any B-biased model gets
~59% accuracy "for free." Rebalancing to 50/50 would make raw accuracy more meaningful and
reduce the gap between biased and debiased metrics.

**Report a "debiased accuracy" metric** alongside raw accuracy in the benchmark results table.
This would be the permutation-averaged logit accuracy, giving a fairer cross-model comparison
that separates perceptual ability from response bias.

### Bottom line

**The 3B model likely cannot perceive chirality at all.** The ~63% self-consistency result was
a statistical artifact of bias sampling + unbalanced labels. The path forward is either:

1. **Model scaling (7B+):** Larger models may have sufficient visual resolution for chirality.
2. **Chirality-specific fine-tuning:** GRPO/DPO on rotation-vs-mirror examples with debiased rewards.
3. **Accept the limitation:** This task may be beyond current 3B VLMs. Focus the benchmark on
   tasks these models can meaningfully perform, and reserve mental rotation for larger models.
4. **Logit-based evaluation as standard:** Regardless of model size, adopt permutation-averaged
   logit accuracy as the primary metric for all forced-choice VLM benchmarks.

## Qwen2.5-VL-7B-Instruct — Debiasing Results

To test whether **model scaling** resolves the chirality problem, the same four
debiasing strategies were evaluated on `Qwen2.5-VL-7B-Instruct`.

### Accuracy summary (Qwen2.5-VL-7B, 83 trials)

| Strategy | N | Correct | Accuracy | p-value | Significant |
|---|---|---|---|---|---|
| D1_logit_single | 83 | 49 | **59.0%** | 0.0619 | No |
| D2_binary_mirror | 83 | 39 | 47.0% | 0.7448 | No |
| D1_logit_perm | 83 | 38 | 45.8% | 0.8100 | No |
| D2_binary_same | 83 | 38 | 45.8% | 0.8100 | No |

### Response bias (7B)

| Strategy | N | A count | B count | A% | chi² p | Biased |
|---|---|---|---|---|---|---|
| D1_logit_perm | 83 | 27 | 56 | 33% | 0.001 | **Yes** |
| D1_logit_single | 83 | 2 | 81 | 2% | 0.000 | **Yes** |
| D2_binary_same | 83 | 39 | 44 | 47% | 0.583 | No |
| D2_binary_mirror | 83 | 38 | 45 | 46% | 0.442 | No |

### Logit permutation analysis (7B)

| Metric | Value |
|---|---|
| Mean P(original A) | 0.499 (0.5 = no bias) |
| Std P(original A) | 0.024 |
| Trials P(A) > 0.6 | 0/83 |
| Trials P(A) < 0.4 | 0/83 |
| Uncertain (0.45-0.55) | **83/83 (100%)** |
| Pass1 & Pass2 agree | 6/83 (7%) |

The 7B model is **even more uncertain** than the 3B: all 83 trials fall in the
0.45–0.55 uncertainty band, and the two permutation passes agree on only 7% of
trials (vs 14% for 3B).

### Accuracy by correct label (7B)

| Strategy | correct=A | correct=B |
|---|---|---|
| D1_logit_perm | 23.5% (8/34) | 61.2% (30/49) |
| D1_logit_single | 2.9% (1/34) | 98.0% (48/49) |
| D2_binary_same | 41.2% (14/34) | 49.0% (24/49) |
| D2_binary_mirror | 41.2% (14/34) | 51.0% (25/49) |

The pattern is identical to the 3B model: `D1_logit_single` is a near-perfect
"always B" strategy (98% on B-correct trials, 3% on A-correct). Debiased
strategies fall to ~46%.

### Accuracy by trial type (7B)

| Strategy | 2D silhouettes | 3D blocks |
|---|---|---|
| D1_logit_perm | 51.0% | 38.2% |
| D1_logit_single | 63.3% | 52.9% |
| D2_binary_mirror | 53.1% | 38.2% |
| D2_binary_same | 40.8% | 52.9% |

### Cross-scale comparison (3B vs 7B)

| Strategy | 3B Accuracy | 7B Accuracy | Δ |
|---|---|---|---|
| D1_logit_single | 56.6% | 59.0% | +2.4% |
| D1_logit_perm | 51.8% | 45.8% | **−6.0%** |
| D2_binary_same | 59.0% | 45.8% | **−13.2%** |
| D2_binary_mirror | 50.6% | 47.0% | −3.6% |

**The 7B model performs worse than the 3B after debiasing.** While the 7B's raw
single-pass accuracy (59.0%) looks marginally better, this is driven entirely by
a stronger B-bias. Once debiased, the 7B model is at or below chance on every
strategy.

### Conclusions on model scaling

1. **Scaling from 3B to 7B does not resolve chirality perception.** The 7B model
   has even stronger position bias (only 7% agreement between permutation passes,
   vs 14% for 3B) and worse debiased accuracy.

2. **The 7B single-pass result (59.0%) is a bias artifact.** It aligns perfectly
   with the "always-B" baseline of 59.0% (49/83). The model answers B on 81/83
   trials.

3. **Debiased accuracy is strictly below chance for 7B.** D1_logit_perm at 45.8%
   and D2_binary_same at 45.8% suggest the model may even have a slight
   *anti-chirality* signal — getting the answer wrong more than expected by chance.

4. **The chirality limitation is architectural, not parametric.** Doubling the
   model size does not help because the underlying visual encoder (likely shared
   SigLIP/ViT architecture) may not represent handedness/chirality at all.

5. **Path forward remains fine-tuning or much larger models.** The results
   suggest that 7B-class Qwen VLMs are fundamentally incapable of this task.
   Improvement requires either chirality-specific fine-tuning (GRPO/DPO with
   debiased reward signals) or significantly larger models (e.g., 72B) where the
   visual representations may capture finer spatial features.

## Statistical Validation of S6 Self-Consistency Result

The S6 self-consistency strategy (k=5, temperature sampling + majority vote) achieved
62.7% accuracy (52/83), the only result statistically significant at p<0.05 in the
text-generation experiments. But is this real signal or a bias artifact amplified by
majority voting?

### Key observations from the raw votes

| Metric | Value |
|---|---|
| Observed B-rate in votes | 286/415 = **68.9%** |
| Accuracy when correct=A | 9/34 = **26.5%** |
| Accuracy when correct=B | 43/49 = **87.8%** |
| Per-trial B-rate std | 0.21 (varies from 0.2 to 1.0) |

The per-vote B-rate is 68.9%. With k=5 majority voting, this amplifies to an
effective B-rate of **P(majority=B) = 82.2%**, closely matching the ~82% we
observed in single-pass text generation.

### Option 1: Biased-coin expected accuracy

A model that picks B with probability 68.9% per vote, with k=5 majority voting:

- **Single draw**: E[acc] = 0.689 × (49/83) + 0.311 × (34/83) = **53.4%**
- **k=5 majority vote**: E[acc] = 0.822 × 0.590 + 0.178 × 0.410 = **55.8%**
- **Actual S6**: **62.7%**
- **Excess**: +6.8%
- **Z-score**: 1.63, **p = 0.052** (marginally non-significant)

### Critical finding: Maximum achievable by ANY biased coin

| B-rate | E[accuracy] with k=5 MV |
|---|---|
| 0% | 41.0% |
| 50% | 50.0% |
| 70% | 56.1% |
| 80% | 58.0% |
| 90% | 58.9% |
| 100% | 59.0% |

**Maximum achievable accuracy by any constant-bias model + k=5 MV: 59.0%**

The observed 62.7% exceeds this theoretical maximum! However, this assumes i.i.d.
bias (same B-rate every trial). In reality, the per-trial B-rate varies
substantially (std=0.21).

### Option 3: Monte Carlo permutation test

**I.I.D. simulation** (constant p_B=68.9%, 10,000 simulations):
- Null distribution: mean=55.8%, std=4.3%
- 95th percentile: 62.7%
- **p-value = 0.070** (not significant at p<0.05)

**Non-i.i.d. simulation** (using each trial's observed B-rate, 10,000 simulations):
- Null distribution: mean=58.8%, std=3.7%
- 95th percentile: 65.1%
- **p-value = 0.183** (clearly not significant)

The non-i.i.d. simulation is the correct null model: it uses the per-trial bias
rates actually observed in the data, accounting for the fact that some trials
trigger stronger bias than others. Under this model, 62.7% is well within the
null distribution.

### Verdict

**The S6 self-consistency result of 62.7% is a bias artifact.** The mechanism:

1. The model has strong B-bias (~69% per vote)
2. k=5 majority voting amplifies this to ~82% effective B-rate
3. The dataset has 59% B-correct labels
4. Per-trial bias variation (std=0.21) creates enough variance that 62.7% falls
   within the null distribution (p=0.183)
5. The apparent significance (p=0.014 from a naive binomial test) was misleading
   because it didn't account for the bias amplification mechanism

**No current strategy — text generation, logit extraction, self-consistency,
permutation averaging, binary decomposition, or any combination — has produced
statistically significant above-chance accuracy after controlling for position
bias, at either the 3B or 7B model scale.**